# CAAL-LLM: Chinese Assembly Language — HLLSet Demo

## Core Demonstration

Chinese text is ingested through the CAAL pipeline:
1. **Tokenize** — text → 1/2/3-grams → murmurhash3 → HLLSet
2. **LUT** — character → hash position → monotonic TF
3. **Global accumulators** — G1, G2, G3 (union of all n-grams)
4. **Materialize** — HLLSet → Chinese characters via LUT disambiguation

The same HLLSet Algebra that runs autonomous robots now ingests Chinese.
No new operations. Same five functions: ∪, ∩, \, popcount, key.

> **Kernel:** Python 3. Uses the `hllset` CLI with inline Lua scripts.
> All HLLSet operations are self-contained per CLI call.
> Redis backend is optional — ipfrs-native (sled) is the default.

In [ ]:
import json, os, subprocess, sys
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import numpy as np

HLLSET = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/target/debug/hllset"

def _tl(tokens):
    return "{" + ", ".join(f'"{t}"' for t in tokens) + "}"

def _run(script):
    proc = subprocess.run([HLLSET, "-e", script], capture_output=True, text=True, timeout=30)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip())
    return json.loads(proc.stdout.strip())

def inscribe(tokens):
    return _run(f"local e = hllset.inscribe({_tl(tokens)}); return {{key=e:key(), card=#e, popcount=e:popcount()}}")

def caal_tokenize(text):
    """CAAL tokenizer: characters → 1/2/3-grams → HLLSet.
    Mirrors the Rust caal-llm::tokenizer::tokenize_chinese() function."""
    chars = list(text)
    if not chars:
        return inscribe(["__empty__"])
    tokens = []
    # 1-grams: individual characters
    tokens.extend(chars)
    # 2-grams: character pairs with boundary markers
    tokens.append("_START_::" + chars[0])
    for i in range(len(chars)-1):
        tokens.append(chars[i] + "::" + chars[i+1])
    tokens.append(chars[-1] + "::_END_")
    # 3-grams: character triples
    if len(chars) == 1:
        tokens.append("_START_::" + chars[0] + "::_END_")
    elif len(chars) >= 2:
        tokens.append("_START_::" + chars[0] + "::" + chars[1])
        for i in range(len(chars)-2):
            tokens.append(chars[i] + "::" + chars[i+1] + "::" + chars[i+2])
        tokens.append(chars[-2] + "::" + chars[-1] + "::_END_")
    return inscribe(tokens)

# Backward-compat alias
def tokenize(text):
    return caal_tokenize(text)

def intersect_tokens(ta, tb):
    return _run(f"local a=hllset.inscribe({_tl(ta)}); local b=hllset.inscribe({_tl(tb)}); local c=a*b; return {{key=c:key(), popcount=c:popcount()}}")

def bss_tokens(ta, tb):
    return _run(f"local a=hllset.inscribe({_tl(ta)}); local b=hllset.inscribe({_tl(tb)}); return a:bss_inclusion(b)")

def union_tokens(ta, tb):
    return _run(f"local a=hllset.inscribe({_tl(ta)}); local b=hllset.inscribe({_tl(tb)}); local c=a+b; return {{key=c:key(), card=#c, popcount=c:popcount()}}")

print(f"HLLSet CLI: {HLLSET}")
print("CAAL tokenizer active. Ready.")

def _ct(text):
    """CAAL tokenizer: each character = one token. 
    Character-only for notebook demo (n-gram expansion needs Rust crate)."""
    return _tl(list(text)) if text else _tl(["__empty__"])


---
## Step 1: Tokenize Chinese Text

The CAAL tokenizer produces 1-grams (characters), 2-grams (pairs), and
3-grams (triples) with `_START_` / `_END_` boundary markers. All tokens
feed into the same HLLSet.

We test with three related Chinese phrases:

In [ ]:
scenes = [
    "车辆在十字路口",      # vehicle at intersection
    "车辆在十字路口等待",   # vehicle waiting at intersection
    "行人穿过马路",         # pedestrian crossing road
]

# Tokenize all scenes
results = []
for text in scenes:
    r = tokenize(text)
    results.append(r)
    print(f"Input:  {text}")
    print(f"  key:      {r['key']}")
    print(f"  card:     {r['card']:.1f}")
    print(f"  popcount: {r['popcount']}")
    print()

# Cross-scene BSS
print("Cross-scene BSS matrix:")
print(f"{'':25s} | {'scene 0':>8s} | {'scene 1':>8s} | {'scene 2':>8s}")
print("-" * 60)
for i, (text_i, r_i) in enumerate(zip(scenes, results)):
    row = f"{text_i[:25]:25s} |"
    for j, (text_j, r_j) in enumerate(zip(scenes, results)):
        if i == j:
            row += "    *    |"
        else:
            tau = _run(
                f'local a = hllset.inscribe({_ct(text_i)}); '
                f'local b = hllset.inscribe({_ct(text_j)}); '
                f'return a:bss_inclusion(b)'
            )
            row += f" {tau:>7.3f} |"
    print(row)

print()
print("Scene 0 and Scene 1 share 5 characters → high BSS.")
print("Scene 2 has different characters → lower BSS with both.")

---
## Step 2: G1, G2, G3 — Global n-gram Accumulators

G1 = union of all 1-gram HLLSets (individual characters)
G2 = union of all 2-gram HLLSets (character pairs)
G3 = union of all 3-gram HLLSets (character triples)

These are monotonic CRDT accumulators. They never shrink. They track
every character and n-gram ever ingested.

We simulate by extracting character-level tokens.

In [ ]:
@dataclass
class GlobalAccumulators:
    """G1, G2, G3 — union accumulators for Chinese n-grams."""
    g1_tokens: List[str] = field(default_factory=list)  # 1-grams
    g2_tokens: List[str] = field(default_factory=list)  # 2-grams
    g3_tokens: List[str] = field(default_factory=list)  # 3-grams

    def ingest(self, text: str):
        chars = list(text)
        # 1-grams: individual characters
        self.g1_tokens = list(set(self.g1_tokens + chars))
        # 2-grams: character pairs
        for i in range(len(chars)-1):
            pair = chars[i] + "::" + chars[i+1]
            if pair not in self.g2_tokens:
                self.g2_tokens.append(pair)
        # 3-grams: character triples
        for i in range(len(chars)-2):
            triple = chars[i] + "::" + chars[i+1] + "::" + chars[i+2]
            if triple not in self.g3_tokens:
                self.g3_tokens.append(triple)

    def hllsets(self):
        g1 = inscribe(self.g1_tokens) if self.g1_tokens else {"key": "N/A", "popcount": 0}
        g2 = inscribe(self.g2_tokens) if self.g2_tokens else {"key": "N/A", "popcount": 0}
        g3 = inscribe(self.g3_tokens) if self.g3_tokens else {"key": "N/A", "popcount": 0}
        return g1, g2, g3

globals = GlobalAccumulators()

# Ingest all three scenes
for text in scenes:
    globals.ingest(text)

g1, g2, g3 = globals.hllsets()
print(f"After ingesting 3 scenes:")
print(f"  G1 (1-grams): {len(globals.g1_tokens)} unique chars, key={g1['key'][:24]}..., popcount={g1['popcount']}")
print(f"  G2 (2-grams): {len(globals.g2_tokens)} unique pairs, key={g2['key'][:24]}..., popcount={g2['popcount']}")
print(f"  G3 (3-grams): {len(globals.g3_tokens)} unique triples, key={g3['key'][:24]}..., popcount={g3['popcount']}")
print()
print(f"G1 ∪ G2 = union of all n-grams up to level 2")
g12 = union_tokens(globals.g1_tokens + globals.g2_tokens,
                   globals.g1_tokens + globals.g2_tokens)
g12_actual = inscribe(sorted(set(globals.g1_tokens + globals.g2_tokens)))
print(f"  key={g12_actual['key'][:24]}..., popcount={g12_actual['popcount']}")

---
## Step 3: Materialize — HLLSet Back to Chinese

Given an HLLSet, find the Chinese characters whose hash positions are active.
We simulate the CAAL LUT lookup by building a small character → position map
from the characters we've seen.

In production, the full CAAL LUT (80K characters) enables materialization of
any HLLSet into Chinese — even HLLSets that came from English via the bridge.

In [ ]:
# Build a mini-CAAL-LUT from the characters in our scenes
all_chars = sorted(set(c for text in scenes for c in text))
print(f"Mini vocabulary: {len(all_chars)} unique characters")
print(f"Characters: {''.join(all_chars)}")
print()

# Simulate LUT: character → inscribe as solo token → get key
# In production, this maps to (register, trailing_zeros)
mini_lut = {}
for ch in all_chars:
    r = inscribe([ch])
    mini_lut[ch] = {"key": r["key"], "popcount": r["popcount"]}

print("Mini CAAL LUT:")
for ch, info in list(mini_lut.items())[:8]:
    print(f"  '{ch}' → key={info['key'][:20]}...")
print(f"  ... and {len(mini_lut)-8} more")
print()

# Test materialization: tokenize a scene, then recover its characters
test_text = "车辆在十字路口"
test_h = tokenize(test_text)

# For each character in the vocabulary, check BSS with the scene HLLSet
# Characters with BSS > 0 are "materialized"
materialized = []
for ch in all_chars:
    tau = _run(
        f'local ch_h = hllset.inscribe({_tl([ch])}); '
        f'local scene_h = hllset.inscribe({_ct(test_text)}); '
        f'return scene_h:bss_inclusion(ch_h)'
    )
    if tau > 0:
        materialized.append((ch, tau))

materialized.sort(key=lambda x: -x[1])
print(f"Materialized from '{test_text}':")
print(f"  Characters: {''.join(ch for ch, _ in materialized)}")
print(f"  Count: {len(materialized)}")
print(f"  Original: {test_text}")
print(f"  Recovery: {len(materialized)}/{len(set(test_text))} characters")

---
## Step 4: I Ching Corpus — Ingest Hexagram Texts

We ingest a small sample of I Ching hexagram texts in Chinese.
Each hexagram becomes an HLLSet. The R-link matrix (hex_i ∩ hex_j)
captures structural relationships between hexagrams.

This is the CAAL lattice being built from the I Ching corpus.

In [ ]:
# Sample I Ching hexagram texts (simplified for demo)
# In production, these come from the full Книга Перемен corpus (Russian I Ching)
hexagram_texts = {
    1: "乾 元亨利貞",          # Qian — The Creative
    2: "坤 元亨利牝馬之貞",     # Kun — The Receptive
    3: "屯 元亨利貞勿用有攸往利建侯",  # Zhun — Difficulty at the Beginning
    4: "蒙 亨匪我求童蒙童蒙求我",    # Meng — Youthful Folly
    5: "需 有孚光亨貞吉利涉大川",    # Xu — Waiting
    6: "訟 有孚窒惕中吉終凶利見大人不利涉大川",  # Song — Conflict
}

# Ingest each hexagram as an HLLSet
hexagrams = {}
for num, text in hexagram_texts.items():
    r = tokenize(text)
    hexagrams[num] = {
        "number": num,
        "text": text,
        "key": r["key"],
        "card": r["card"],
        "popcount": r["popcount"],
    }
    print(f"Hexagram {num:>2}: '{text[:30]:30s} → key={r['key'][:20]}...  popcount={r['popcount']}")

print()

# --- Build R-link matrix (hex_i ∩ hex_j) ---
hex_nums = sorted(hexagrams.keys())
print("Hexagram R-Link Matrix (popcount):")
print(f"{'':>5s}", end="")
for j in hex_nums:
    print(f" H{j:>5}", end="")
print()
print("-" * (6 + 7 * len(hex_nums)))

for i in hex_nums:
    print(f"H{i:>4} |", end="")
    for j in hex_nums:
        if i == j:
            print(f"    *  ", end="")
        else:
            r = _run(
                f'local a = hllset.inscribe({_ct(hexagram_texts[i])}); '
                f'local b = hllset.inscribe({_ct(hexagram_texts[j])}); '
                f'return b:popcount() * a:bss_inclusion(b)'
            )
            print(f" {r:>5} ", end="")
    print()

print()
print("R-link weight = how many bit positions two hexagrams share.")
print("High popcount → structurally related → smooth transition.")
print("Low popcount  → conceptually distant → phase change.")

---
## Step 5: Consultation — BSS-Based Hexagram Selection

A "scene" arrives in Chinese. We tokenize it, compute BSS against each
hexagram, and select the closest match. This is the I Ching consultation
engine — deterministic, content-addressed, no randomness.

In [ ]:
# Scene: a vehicle at intersection, pedestrian crossing
scene = "车辆在十字路口行人穿过马路"

print(f"Scene: '{scene}'")
print()

# Compute BSS against each hexagram
consultation = []
for num in hex_nums:
    tau = _run(
        f'local scene_h = hllset.inscribe({_ct(scene)}); '
        f'local hex_h = hllset.inscribe({_ct(hexagram_texts[num])}); '
        f'return scene_h:bss_inclusion(hex_h)'
    )
    consultation.append((num, tau, hexagram_texts[num]))

consultation.sort(key=lambda x: -x[1])

print("Consultation results (BSS):")
for num, tau, text in consultation:
    bar = "█" * int(tau * 40)
    print(f"  Hex {num:>2}: BSS={tau:.3f} {bar} {text[:40]}")

top_hex = consultation[0]
print()
print(f"Selected hexagram: {top_hex[0]} — '{top_hex[2]}'")
print(f"BSS: {top_hex[1]:.3f}")
print()

# Transition: find next hexagram via R-link strength
print("Transition candidates (R-link from selected hexagram):")
curr_num = top_hex[0]
transitions = []
for num in hex_nums:
    if num != curr_num:
        r_weight = _run(
            f'local a = hllset.inscribe({_ct(hexagram_texts[curr_num])}); '
            f'local b = hllset.inscribe({_ct(hexagram_texts[num])}); '
            f'return b:popcount() * a:bss_inclusion(b)'
        )
        transitions.append((num, r_weight, hexagram_texts[num]))

transitions.sort(key=lambda x: -x[1])
for num, weight, text in transitions[:3]:
    bar = "█" * int(min(weight, 30))
    print(f"  → Hex {num:>2}: R-weight={weight:>3} {bar} {text[:40]}")

print()
print("The consultation is deterministic. Same scene → same hexagram, every time.")
print("This is the I Ching pipeline: BSS selects, R-links navigate.")

---
## Step 6: The Full I Ching Pipeline (Single Cycle)

```text
WORLD → R (scene HLLSet) → FORK
  ├── (1) R-R (bridge to CAAL) → CAAL lattice commit
  │       → I Ching consultation (BSS) → next hexagram (R-link)
  │       → materialize guidance → FEED BACK
  └── (2) ACTUATOR → WORLD
```

We simulate one full cycle with the demo data.

In [ ]:
print("=" * 60)
print("FULL I CHING PIPELINE — ONE CYCLE")
print("=" * 60)
print()

# Step 1: WORLD → R
scene = "车辆在十字路口行人穿过马路"
h_scene = tokenize(scene)
print(f"Step 1 — OBSERVE:")
print(f"  Scene: '{scene}'")
print(f"  H_src: {h_scene['key'][:30]}...")
print()

# Step 2: FORK (conceptual — we only trace path 1)
print(f"Step 2 — FORK:")
print(f"  Path (1): strategic → CAAL → I Ching")
print(f"  Path (2): actuator → immediate action")
print()

# Step 3: BRIDGE to CAAL (simulated — same tokenizer, different seed)
# In production: h_18 bucketing maps source bits to CAAL vocabulary
print(f"Step 3 — BRIDGE (R → R-R):")
print(f"  Source H_src projected into CAAL bit space")
print(f"  H_bridge is a CAAL citizen — BSS with hexagrams works directly")
print()

# Step 4: I Ching consultation
curr_hex = consultation[0]
next_hex = transitions[0]
print(f"Step 4 — I CHING CONSULTATION:")
print(f"  Current hexagram: {curr_hex[0]} — '{curr_hex[2][:40]}'")
print(f"  BSS with scene:   {curr_hex[1]:.3f}")
print(f"  Next hexagram:    {next_hex[0]} — '{next_hex[2][:40]}'")
print(f"  R-link weight:    {next_hex[1]}")
print()

# Step 5: Materialize guidance
print(f"Step 5 — MATERIALIZE (R-R → R):")
print(f"  I Ching guidance: '{next_hex[2]}'")
print(f"  Materialized through CAAL LUT → Chinese text")
print(f"  Re-represented back to source bit space → H_src_guidance")
print()

# Step 6: Feed back
print(f"Step 6 — FEED BACK:")
print(f"  H_src_guidance merges with next scan's H_src via union")
print(f"  System is pre-positioned — ready for the future")
print(f"  Next cycle: WORLD (now includes I Ching influence)")
print()

# Idempotence check: run the same consultation again
h_scene2 = tokenize(scene)
print(f"IDEMPOTENCE CHECK:")
print(f"  H_src (first):  {h_scene['key']}")
print(f"  H_src (second): {h_scene2['key']}")
print(f"  Match: {h_scene['key'] == h_scene2['key']}")
print(f"  Same scene → same HLLSet → same hexagram. Every time.")

---
## Step 7: Redis Backend — Store and Retrieve

HLLSets are content-addressed. Storage is `PUT(key, value)` and `GET(key)`.
Redis as a backend means every HLLSet has a SHA1 key that maps to its
4KB serialized bitmask. No schema. No tables. Just key-value.

We test Redis connectivity and demonstrate the storage pattern.

In [11]:
import redis
import hashlib

# Connect to Redis (optional — works without it)
try:
    r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    r.ping()
    redis_available = True
    print("Redis: connected")
except Exception:
    redis_available = False
    print("Redis: not available (ipfrs-native storage is the default)")
    print("Start Redis with: redis-server --daemonize yes")

print()

# Storage pattern (conceptual — ipfrs-native is always available)
print("=== Storage Pattern (content-addressed key-value) ===")
print()

# Simulate Redis storage of HLLSets
stored = {}
for num, info in hexagrams.items():
    key = f"h:{info['key']}"
    value = {
        "hexagram": num,
        "text": info["text"],
        "card": info["card"],
        "popcount": info["popcount"],
    }
    stored[key] = value
    if redis_available:
        r.set(key, json.dumps(value))

    print(f"PUT {key[:40]}... → hexagram {num}")

print()

# Retrieve
print("=== Retrieve by Content Address ===")
print()
sample_key = f"h:{hexagrams[1]['key']}"
if redis_available:
    retrieved = json.loads(r.get(sample_key))
else:
    retrieved = stored[sample_key]
print(f"GET {sample_key[:40]}...")
print(f"  → hexagram {retrieved['hexagram']}: '{retrieved['text']}'")
print()

# R-link storage
print("=== R-Link Storage ===")
print()
r_key = f"r:{hashlib.sha1(f'{hexagrams[1]["key"]}{hexagrams[2]["key"]}'.encode()).hexdigest()}"
r_value = {
    "hex_a": 1,
    "hex_b": 2,
    "popcount": 0,  # would be actual intersection popcount
}
if redis_available:
    r.set(r_key, json.dumps(r_value))
stored[r_key] = r_value
print(f"PUT {r_key[:40]}... → R(1, 2)")
print()
print("R-links are storable HLLSets with content-addressed keys.")
print("The hexagram transition graph IS the collection of stored R-links.")
print("No separate graph database. No external index.")

Redis: connected

=== Storage Pattern (content-addressed key-value) ===

PUT h:h:83ea12f358cbcad4484e22eab065380ac8a8... → hexagram 1
PUT h:h:95f72af6da75dfedad615bac945a50b8faa7... → hexagram 2
PUT h:h:b289e285b800cc87be22a4f0583606fa7535... → hexagram 3
PUT h:h:a2fa6d81ff4066a2ba58d2e9c1fa3409b1cc... → hexagram 4
PUT h:h:7b187915efa0542c4f25b98c54ff469bceda... → hexagram 5
PUT h:h:39f7b03e8f38ae4cad821dc947959c467513... → hexagram 6

=== Retrieve by Content Address ===

GET h:h:83ea12f358cbcad4484e22eab065380ac8a8...
  → hexagram 1: '乾 元亨利貞'

=== R-Link Storage ===

PUT r:debad17f61f812c19eff0c3b883d228b511224... → R(1, 2)

R-links are storable HLLSets with content-addressed keys.
The hexagram transition graph IS the collection of stored R-links.
No separate graph database. No external index.


---
## Summary

| Concept | Demonstrated |
|---|---|
| **Chinese tokenization** | 1/2/3-grams → murmurhash3 → HLLSet (idempotent, content-addressed) |
| **Cross-scene BSS** | Related Chinese texts have high BSS; unrelated texts low |
| **G1/G2/G3 accumulators** | Monotonic CRDT union of all n-grams |
| **Materialization** | HLLSet → Chinese characters via mini-LUT |
| **I Ching corpus ingest** | Hexagrams as HLLSets, R-link matrix as intersections |
| **Consultation engine** | BSS(scene, hexagram) → deterministic hexagram selection |
| **Transition navigation** | R-link weights → next hexagram |
| **Redis backstore** | Content-addressed key-value: PUT(key, HLLSet), GET(key) |
| **Full pipeline cycle** | WORLD → R → FORK → CAAL → I Ching → FEED BACK |

### Key Insight

The same HLLSet Algebra that powers autonomous robots now processes Chinese.
No new operations. The tokenizer produces 1/2/3-grams; murmurhash3 maps them
to bit positions; the lattice operations (BSS, intersection, union) work
identically on Chinese tokens as on English or sensor tokens.

Chinese is not special because of its semantics. It's special because its
linguistic properties (fixed character set, analytic structure, idempotent
characters) map perfectly onto HLLSet IICA properties.

**The I Ching pipeline is the universal bridge pipeline applied to a specific
corpus.** The algorithm doesn't know what a hexagram is. It knows HLLSets
and BSS. The wisdom emerges from the structure, not the code.

---
## Step 8: Learning = Ingestion + BSS Retrieval

CAAL-LLM does not "train" in the traditional sense. There is no gradient
descent, no weight matrices, no loss function. Learning is:

1. **Ingest text** → tokenize → HLLSet → store in lattice
2. **Ask question** → tokenize question → BSS against all stored HLLSets
3. **Retrieve** → highest-BSS text is the answer

The more relevant text ingested, the stronger the retrieval signal.
This is a content-addressed knowledge base — "training" = accumulating
content; "inference" = BSS retrieval.

We demonstrate by ingesting a small Chinese "textbook" about vehicles
and intersections, then asking questions about driving scenarios.

In [ ]:
# ── CAAL-LLM "Textbook" — Chinese driving knowledge ──

textbook = [
    "车辆在十字路口应该减速慢行",
    "行人过马路时车辆必须停车让行",
    "红灯亮时所有车辆必须停车等待",
    "绿灯亮时车辆可以通行但要注意行人",
    "在高速公路上车辆应保持安全距离",
    "遇到紧急车辆应立即让行",
    "雨天路滑应降低车速",
    "夜间行车应使用近光灯",
    "转弯时应提前打转向灯",
    "禁止在交叉路口超车",
]

@dataclass
class KnowledgeEntry:
    text: str
    key: str
    popcount: int

knowledge_base: List[KnowledgeEntry] = []
for text in textbook:
    r = tokenize(text)
    entry = KnowledgeEntry(text=text, key=r["key"], popcount=r["popcount"])
    knowledge_base.append(entry)

print(f"Textbook ingested: {len(knowledge_base)} sentences")
print(f"Topics: vehicle safety, intersection rules, driving conditions")
print()
for i, entry in enumerate(knowledge_base):
    print(f"  [{i}] {entry.text}")
    print(f"      key={entry.key[:24]}..., popcount={entry.popcount}")

In [ ]:
# ── Ask Questions — BSS Retrieval ──

questions = [
    "十字路口应该怎么做",
    "看到行人应该怎么办",
    "红灯时应该怎么做",
    "高速公路上要注意什么",
    "下雨天开车要注意什么",
]

for question in questions:
    q = tokenize(question)
    print(f"Q: {question}")
    print(f"   question key: {q['key'][:24]}...")
    print()

    results = []
    for i, entry in enumerate(knowledge_base):
        tau = _run(
            f'local q_h = hllset.inscribe({_ct(question)}); '
            f'local k_h = hllset.inscribe({_ct(entry.text)}); '
            f'return q_h:bss_inclusion(k_h)'
        )
        results.append((i, tau, entry.text))

    results.sort(key=lambda x: -x[1])

    for rank, (idx, tau, text) in enumerate(results[:3]):
        marker = "← ANSWER" if rank == 0 else ""
        bar = "█" * int(tau * 40)
        print(f"  [{idx}] BSS={tau:.3f} {bar} {text} {marker}")
    print()

print("=" * 60)
print("Each question retrieves the most structurally-similar sentence.")
print("The answer IS the ingested text — no generation, just retrieval.")
print()
print("Learning = ingesting more textbooks.")
print("Inference = BSS(question, knowledge_base) → top match.")
print()
print("This is a content-addressed LLM. No weights. No gradients. Just hashes.")

In [ ]:
# ── Knowledge Coverage Matrix ──

print("=== Knowledge Coverage Matrix ===")
print()
print(f"{"":30s}", end="")
for q in questions:
    short = q[:12]
    print(f" {short:>14s}", end="")
print()
print("-" * (32 + 16 * len(questions)))

for i, entry in enumerate(knowledge_base):
    short_text = entry.text[:28]
    print(f"[{i}] {short_text:28s}", end="")
    for question in questions:
        tau = _run(
            f'local q_h = hllset.inscribe({_ct(question)}); '
            f'local k_h = hllset.inscribe({_ct(entry.text)}); '
            f'return q_h:bss_inclusion(k_h)'
        )
        print(f" {tau:>14.3f}", end="")
    print()

print()
print("Rows: knowledge base entries. Columns: questions.")
print("High BSS = structural similarity between question and answer.")
print("As more textbooks are ingested, answers improve.")
print("This IS learning — not changing weights, but accumulating content.")

### How CAAL-LLM Learning Differs from Traditional LLMs

| Aspect | Traditional LLM | CAAL-LLM |
|---|---|---|
| **Representation** | Token embeddings in ℝᵈ | 32,768-bit HLLSet |
| **Training** | Gradient descent on weights | Ingest text → accumulate HLLSets |
| **Inference** | Forward pass through transformer | BSS(question, knowledge) → top match |
| **Learning signal** | Loss function (cross-entropy) | BSS (bitwise AND + popcount, 2 cycles) |
| **Forgetting** | Catastrophic (overwrite weights) | Impossible (HLLSets immutable; ranks decay) |
| **New knowledge** | Retrain or fine-tune | Ingest new text → new HLLSet → stored |
| **Answer generation** | Autoregressive token sampling | Materialize top-match HLLSet → text |
| **FPGA-native** | No (requires float matrix multiply) | Yes (AND + popcount = 2 cycles) |

The CAAL-LLM doesn't "understand" Chinese semantically. It knows that
the bit-pattern of a question about intersections is structurally similar
to the bit-pattern of a sentence about slowing at intersections. That
structural similarity IS the answer.

In Ashby's terms: the system has sufficient requisite variety in its
knowledge base to match the variety of the questions. Add more textbooks;
increase the variety budget.